# 🔍 Aula 19 — Interpretabilidade e Explicabilidade (XAI)**Disciplina:** IA Aplicada à Engenharia Química**Dataset:** cstr_exotermico_xai.csv — rendimento de CSTR exotérmico---## O que vamos fazerAbrir a "caixa-preta" do XGBoost: *por que o modelo fez esta predição?*- Importância global (SHAP summary plot)- Dependência por feature (dependence plot)- Explicação local (waterfall plot)- **Validar a consistência física** com a termodinâmica do reator

## Contexto físico (arbitro final)Reator **exotérmico**: conversão máxima em T_ótimo (~105 °C).- T abaixo do ótimo → reação lenta; T acima → equilíbrio desloca p/ reagentes- Curva em **sino** de rendimento vs T- Pressão alta favorece (fase gasosa, Le Chatelier)- C_feed alta dilui; vazão alta reduz tempo de residência → conversão cai

## 3.1 — Exercício Guiado: SHAP no reatorSiga as células. Pipeline: treinar → explainer → summary → dependence → waterfall → validar fisicamente.

### Passo 1: treinar XGBoost

In [ ]:
import pandas as pd, numpy as npfrom xgboost import XGBRegressorfrom sklearn.model_selection import train_test_splitURL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula19/cstr_exotermico_xai.csv"df = pd.read_csv(URL)feats = ['T_reator_C','pressao_bar','C_feed_mol_L','vazao_L_min']X, y = df[feats], df['rendimento_pct']Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)xgb = XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=42, verbosity=0)xgb.fit(Xtr, ytr)print("Modelo treinado")

### Passo 2: criar explainer + SHAP values

In [ ]:
import shapexplainer = shap.TreeExplainer(xgb)shap_values = explainer(Xte)print(shap_values.shape)

### Passo 3: summary plot (importância global + direcao)

In [ ]:
shap.summary_plot(shap_values, Xte, max_display=4)# Vermelho = valor alto da feature; Azul = baixo# A direcao do SHAP mostra o sinal do efeito na predicao

### Passo 4: dependence plot de T (deve "dobrar")

In [ ]:
shap.dependence_plot('T_reator_C', shap_values.values, Xte)# Fisica: SHAP + ate ~105 C, depois - (sino do equilibrio exotermico)

### Passo 5: dependence plot de P e vazao

In [ ]:
shap.dependence_plot('pressao_bar', shap_values.values, Xte)

### Passo 6: waterfall de 2 predicoes

In [ ]:
# Indice de uma predicao CERTA e uma ERRADA (erro)yp = xgb.predict(Xte)err = np.abs(yp - yte.values)i_certa = int(np.argmin(err))i_errada = int(np.argmax(err))print(f"certa: erro={err[i_certa]:.3f}  errada: erro={err[i_errada]:.3f}")shap.plots.waterfall(shap_values[i_certa])

### Passo 7: waterfall da predicao errada + conclusao

In [ ]:
shap.plots.waterfall(shap_values[i_errada])# O waterfall mostra qual feature afastou a predicao do valor real

> **Conclusao de consistencia fisica:** o SHAP de T replica a curva em sino do> equilibrio exotermico? Os sinais de P (+), C_feed (+) e vazao (-) concordam> com a termodinamica?

---## 3.2 — Exercicio em Grupo: Diagnostico com WaterfallCada grupo investiga 1 predicao SUSPEITA (erro > 5%). O waterfall revela a feature que causou o erro.Checklist: [ ] waterfall gerado, [ ] feature dominante, [ ] sinal vs termodinamica, [ ] hipotese (leakage/overfitting/colinearidade/interacao), [ ] plano de correcao.

## Checklist final- [ ] XGBoost treinado- [ ] explainer + shap_values- [ ] summary plot- [ ] dependence de T, P, vazao- [ ] waterfall de 2 predicoes- [ ] consistencia fisica validada- [ ] diagnostico de inconsistencias